# SDXL Image Generation

توليد الصور بالذكاء الاصطناعي باستخدام Stable Diffusion XL على Kaggle GPU.

المميزات:
- دعم prompts عربية (ترجمة تلقائية للإنجليزية للـ SDXL)
- 3 نسب عرض: 1:1 (مربع) + 9:16 (ستوري) + 4:5 (بورتريه)
- رفع النتائج إلى GitHub Release

In [ ]:
# Cell 2 — تثبيت diffusers + torch
import subprocess, sys
!pip install -q diffusers transformers accelerate torch --index-url https://download.pytorch.org/whl/cu121
!pip install -q sentencepiece Pillow
import torch
from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"}')

In [ ]:
# Cell 3 — تحميل نموذج SDXL
import torch, time
from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler

print('Loading SDXL base 1.0 (~7GB VRAM) ...')
t0 = time.time()
pipe = StableDiffusionXLPipeline.from_pretrained(
    'stabilityai/stable-diffusion-xl-base-1.0',
    torch_dtype=torch.float16,
    variant='fp16',
    use_safetensors=True,
)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to('cuda')
# Memory optimisations for 2× T4 (16GB each)
pipe.enable_model_cpu_offload()
pipe.enable_vae_slicing()
print(f'[OK] SDXL loaded in {time.time()-t0:.1f}s')

In [ ]:
# Cell 4 — توليد صورة من برومبت (يدعم العربية)
import os, re

# Translate Arabic prompt -> English (free heuristic via transformers MarianMT)
# Falls back to the original prompt if translation is unavailable
try:
    from transformers import pipeline as hf_pipeline
    translator = hf_pipeline('translation', model='Helsinki-NLP/opus-mt-ar-en')
    def translate_ar_to_en(text: str) -> str:
        if not re.search(r'[\u0600-\u06FF]', text):
            return text  # already latin
        out = translator(text, max_length=512)
        return out[0]['translation_text'] if out else text
except Exception as e:
    print(f'[WARN] translator unavailable ({e}); using prompt as-is')
    def translate_ar_to_en(text: str) -> str:
        return text

raw_prompt = os.environ.get('SDXL_PROMPT', 'عيادة أسنان حديثة بإضاءة دافئة، تصوير احترافي')
negative    = os.environ.get('SDXL_NEGATIVE', 'blurry, low quality, distorted, watermark, text')
steps        = int(os.environ.get('SDXL_STEPS', '30'))
guidance     = float(os.environ.get('SDXL_GUIDANCE', '7.5'))

english_prompt = translate_ar_to_en(raw_prompt)
print(f'Arabic prompt : {raw_prompt}')
print(f'English prompt: {english_prompt}')
print(f'Negative      : {negative}')
print(f'Steps={steps}, CFG={guidance}')

In [ ]:
# Cell 5 — توليد عدة نسب عرض (1:1, 9:16, 4:5)
import os, time
from PIL import Image

out_dir = '/kaggle/working/output'
os.makedirs(out_dir, exist_ok=True)

ASPECTS = {
    'square'  : (1024, 1024),   # 1:1  — Instagram feed / carousel
    'story'   : (768,  1344),   # 9:16 — Instagram/TikTok story
    'portrait': (864,  1080),   # 4:5  — Instagram portrait
}

seed = int(os.environ.get('SDXL_SEED', '0')) or None
generator = torch.Generator('cuda')
if seed is not None:
    generator.manual_seed(seed)
print(f'Seed: {seed}')

saved = []
for label, (w, h) in ASPECTS.items():
    print(f'\nGenerating {label} ({w}x{h}) ...')
    t0 = time.time()
    img = pipe(
        prompt=english_prompt,
        negative_prompt=negative,
        width=w, height=h,
        num_inference_steps=steps,
        guidance_scale=guidance,
        generator=generator,
    ).images[0]
    print(f'  done in {time.time()-t0:.1f}s')
    path = os.path.join(out_dir, f'sdxl_{label}.png')
    img.save(path, 'PNG', optimize=True)
    saved.append(path)
    print(f'  saved {path}  ({os.path.getsize(path)/1e6:.2f} MB)')

print(f'\n[OK] Generated {len(saved)} images')

In [ ]:
# Cell 6 — رفع الصور إلى GitHub Release
import os
outputs = [p for p in saved if os.path.exists(p)]
print(f'Files to upload: {len(outputs)}')
for p in outputs:
    print(f'  - {p}  ({os.path.getsize(p)/1e6:.2f} MB)')

import os
import json
import subprocess
import urllib.request
import urllib.error

# --- إعدادات الرفع إلى GitHub Release ---
# يتم تمريرها عبر Kaggle Secrets أو متغيرات البيئة
GH_TOKEN = os.environ.get('GH_TOKEN', os.environ.get('GITHUB_TOKEN', ''))
GH_REPO = os.environ.get('GH_REPO', 'mickystudio/micky-infra')
TASK_ID = os.environ.get('KAGGLE_TASK_ID', 'test')
RELEASE_TAG = os.environ.get('RELEASE_TAG', f'task-{TASK_ID}')


def gh_release_api(method: str, path: str, payload: dict | None = None,
                   upload_url: str | None = None, asset_path: str | None = None):
    """Calls the GitHub REST API (or uploads an asset) with auth."""
    headers = {
        'Accept': 'application/vnd.github+json',
        'Authorization': f'Bearer {GH_TOKEN}',
        'X-GitHub-Api-Version': '2022-11-28',
    }
    try:
        if upload_url and asset_path:
            # Upload asset via multipart form
            import mimetypes
            mime = mimetypes.guess_type(asset_path)[0] or 'application/octet-stream'
            headers['Content-Type'] = mime
            with open(asset_path, 'rb') as fh:
                req = urllib.request.Request(upload_url, data=fh.read(), headers=headers, method='POST')
        else:
            data = json.dumps(payload).encode() if payload else None
            base = 'https://api.github.com'
            url = f'{base}{path}' if upload_url is None else upload_url
            req = urllib.request.Request(url, data=data, headers=headers, method=method)
        with urllib.request.urlopen(req, timeout=120) as resp:
            body = resp.read().decode('utf-8', errors='ignore')
            return resp.status, json.loads(body) if body else {}
    except urllib.error.HTTPError as e:
        err_body = e.read().decode('utf-8', errors='ignore')
        print(f'GitHub API error {e.code}: {err_body}')
        return e.code, {}
    except Exception as e:
        print(f'GitHub upload exception: {e}')
        return 0, {}


def ensure_release():
    """Create the release if it doesn't exist, return its upload_url template."""
    # Try fetching first
    status, data = gh_release_api('GET', f'/repos/{GH_REPO}/releases/tags/{RELEASE_TAG}')
    if status == 200:
        upload_url = data.get('upload_url', '').replace('{?name,label}', '')
        return upload_url
    # Create the release
    payload = {
        'tag_name': RELEASE_TAG,
        'name': f'Kaggle task {TASK_ID}',
        'body': 'Auto-uploaded from Kaggle notebook',
        'draft': False,
        'prerelease': False,
    }
    status, data = gh_release_api('POST', f'/repos/{GH_REPO}/releases', payload=payload)
    return data.get('upload_url', '').replace('{?name,label}', '') if data else ''


def upload_asset(upload_url: str, asset_path: str) -> str | None:
    """Uploads a single file to the release. Returns the browser_download_url."""
    name = os.path.basename(asset_path)
    url = f'{upload_url}?name={name}'
    status, data = gh_release_api('POST', '', upload_url=url, asset_path=asset_path)
    if status in (201, 200) and data:
        print(f'  uploaded: {name} -> {data.get("browser_download_url")}')
        return data.get('browser_download_url')
    print(f'  FAILED to upload {name} (status={status})')
    return None


def upload_files_to_release(file_paths, manifest=None):
    """Uploads all files to the GitHub release for this task."""
    if not GH_TOKEN:
        print('WARNING: GH_TOKEN not set. Skipping upload to GitHub Release.')
        print('Files remain available on the Kaggle output at /kaggle/working/output')
        return []
    upload_url = ensure_release()
    if not upload_url:
        print('ERROR: could not create/find GitHub release. Aborting upload.')
        return []
    uploaded = []
    for fp in file_paths:
        if not os.path.exists(fp):
            print(f'  skip (missing): {fp}')
            continue
        url = upload_asset(upload_url, fp)
        if url:
            uploaded.append(url)
    # Optional manifest (JSON describing outputs)
    if manifest:
        manifest_path = '/kaggle/working/manifest.json'
        with open(manifest_path, 'w') as f:
            json.dump(manifest, f, ensure_ascii=False, indent=2)
        upload_asset(upload_url, manifest_path)
    return uploaded


print(f'Release tag: {RELEASE_TAG}')
print(f'Repo:         {GH_REPO}')
print(f'Token set:    {bool(GH_TOKEN)}')

manifest = {
    'task_id': os.environ.get('KAGGLE_TASK_ID', 'test'),
    'prompt_ar': raw_prompt,
    'prompt_en': english_prompt,
    'negative': negative,
    'steps': steps, 'guidance': guidance, 'seed': seed,
    'outputs': [{'file': os.path.basename(p), 'size_mb': round(os.path.getsize(p)/1e6, 2)} for p in outputs],
}
uploaded_urls = upload_files_to_release(outputs, manifest=manifest)
print(f'\n[OK] {len(uploaded_urls)} images uploaded to GitHub Release.')
for u in uploaded_urls:
    print(f'  {u}')

## تم!

الصور جاهزة للاستخدام في وضع Graphics Mode أو للتحرير في WebApp Canvas (P5-T2).